# Collapse Budget


In [38]:
import pandas as pd
import numpy as np

In [39]:
df = pd.read_excel("Docs/Expense_Budget_20260520.xlsx")

In [40]:
print(df.columns)
print(df.dtypes)
df.head(6)

Index(['Publication Date', 'Fiscal Year', 'Agency Number', 'Agency Name',
       'Unit Appropriation Number', 'Unit Appropriation Name',
       'Adopted Budget Position', 'Adopted Budget Amount',
       'Current Modified Budget Position', 'Current Modified Budget Amount',
       'Financial Plan Position', 'Financial Plan Budget Amount'],
      dtype='str')
Publication Date                    int64
Fiscal Year                         int64
Agency Number                         str
Agency Name                           str
Unit Appropriation Number           int64
Unit Appropriation Name               str
Adopted Budget Position             int64
Adopted Budget Amount               int64
Current Modified Budget Position    int64
Current Modified Budget Amount      int64
Financial Plan Position             int64
Financial Plan Budget Amount        int64
dtype: object


,Publication Date,Fiscal Year,Agency Number,Agency Name,Unit Appropriation Number,Unit Appropriation Name,Adopted Budget Position,Adopted Budget Amount,Current Modified Budget Position,Current Modified Budget Amount,Financial Plan Position,Financial Plan Budget Amount
0,20260512,2027,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,319,40515979,319,40316729,322,46751202
1,20260217,2027,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,319,40515979,319,40553729,318,40465085
2,20250630,2026,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,308,38628360,309,37320045,319,40515979
3,20260512,2027,002,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS,0,4237982,0,4459520,0,5031799
4,20260217,2027,002,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS,0,4237982,0,4212520,0,4489982
5,20250630,2026,002,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS,0,4478587,0,4204822,0,4237982


There are three publication dates:

- 20260512 - FY 27 Executive
- 20260217 - FY 27 Prelim
- 20250630 - FY 26 Adopted

For each Agency/Unit of Appropriation, we need to collapse these three rows into a single row, creating the following columns:

| column name          | What to use                                                                      | Choice of source |
| -------------------- | -------------------------------------------------------------------------------- | ---------------- |
| **fy2027_exec:**     | 20260512 Financial Plan Budget Amount                                            | FY 27 Executive  |
| **fy2027_prelim:**   | 20260217 Financial Plan Budget Amount                                            | FY 27 Prelim     |
| **fy2026_modified:** | 20260512/20260217 Current Modified Budget Amount                                 | FY 27 Executive  |
| **fy2026_adopted:**  | 20250630 Financial Plan Budget Amount or 20260512/20260217 Adopted Budget Amount | FY 27 Executive  |
|                      | absolute change from fy2027_exec to fy2027_prelim and to fy2026_adopted          |
|                      | relative change from fy2027_exec to fy2027_prelim and to fy2026_adopted          |


## Set up base dataframe to fill in


In [41]:
base_cols = ['Agency Number', 'Agency Name', 'Unit Appropriation Number', 'Unit Appropriation Name']
base_df = df[base_cols].drop_duplicates()
base_df.head()

,Agency Number,Agency Name,Unit Appropriation Number,Unit Appropriation Name
0,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS
3,002,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS
6,002,MAYORALTY,40,OFFICE OF MGMT AND BUDGET-PS
9,002,MAYORALTY,41,OFFICE OF MGMT AND BUDGET-OTPS
12,002,MAYORALTY,50,CRIMINAL JUSTICE PROGRAMS PS


Create an agency num/name map for later


In [42]:
AGENCY_MAP = df[['Agency Number', 'Agency Name']].drop_duplicates().set_index('Agency Number')['Agency Name'].to_dict()
AGENCY_MAP

{'002': 'MAYORALTY',
 '003': 'BOARD OF ELECTIONS',
 '004': 'CAMPAIGN FINANCE BOARD',
 '008': 'OFFICE OF THE ACTUARY',
 '010': 'BOROUGH PRESIDENT - MANHATTAN',
 '011': 'BOROUGH PRESIDENT BRONX',
 '012': 'BOROUGH PRESIDENT - BROOKLYN',
 '013': 'BOROUGH PRESIDENT - QUEENS',
 '014': 'BOROUGH PRESIDENT STATEN ISLAND',
 '015': 'OFFICE OF THE COMPTROLLER',
 '017': 'DEPARTMENT OF EMERGENCY MANAGEMENT',
 '021': 'OFFICE OF ADMINISTRATIVE TAX APPEALS',
 '025': 'LAW DEPARTMENT',
 '030': 'DEPARTMENT OF CITY PLANNING',
 '032': 'DEPARTMENT OF INVESTIGATION',
 '035': 'NEW YORK RESEARCH LIBRARIES',
 '037': 'NEW YORK PUBLIC LIBRARY',
 '038': 'BROOKLYN PUBLIC LIBRARY',
 '039': 'QUEENS BOROUGH PUBLIC LIBRARY',
 '040': 'DEPARTMENT OF EDUCATION',
 '042': 'CITY UNIVERSITY OF NEW YORK',
 '054': 'CIVILIAN COMPLAINT REVIEW BOARD',
 '056': 'POLICE DEPARTMENT',
 '057': 'FIRE DEPARTMENT',
 '058': 'OFFICE OF COMMUNITY SAFETY',
 '063': "DEPARTMENT OF VETERANS' SERVICES",
 '068': "ADMIN FOR CHILDREN'S SERVICES",
 '06

## Build new columns one at a time


### fy2027_exec


In [43]:
fy2027_exec_cols = ['Agency Number', 'Unit Appropriation Number', "Financial Plan Budget Amount", "Financial Plan Position"]
fy2027_exec_pubdate = 20260512

fy2027_exec = df[df["Publication Date"] == fy2027_exec_pubdate][fy2027_exec_cols].rename(
    columns={
        "Financial Plan Budget Amount":"fy2027_exec_budget_amount",
        "Financial Plan Position":"fy2027_exec_position"
        }
)
fy2027_exec

,Agency Number,Unit Appropriation Number,fy2027_exec_budget_amount,fy2027_exec_position
0,002,20,46751202,322
3,002,21,5031799,0
6,002,40,55900321,467
9,002,41,11666672,0
14,002,61,18597617,193
...,...,...,...,...
2137,944,1,685873,8
2140,944,2,15713,0
2143,945,1,622523,5
2146,945,2,35105,0


### fy2027_prelim


In [44]:
fy2027_prelim_cols = ['Agency Number', 'Unit Appropriation Number', "Financial Plan Budget Amount", "Financial Plan Position"]
fy2027_prelim_pubdate = 20260217

fy2027_prelim = df[df["Publication Date"] == fy2027_prelim_pubdate][fy2027_prelim_cols].rename(
    columns={
        "Financial Plan Budget Amount":"fy2027_prelim_budget_amount",
        "Financial Plan Position":"fy2027_prelim_position",        
        }
    )
fy2027_prelim

,Agency Number,Unit Appropriation Number,fy2027_prelim_budget_amount,fy2027_prelim_position
1,002,20,40465085,318
4,002,21,4489982,0
7,002,40,56067194,467
10,002,41,11349070,0
15,002,61,18110617,188
...,...,...,...,...
2144,945,1,624523,5
2147,945,2,31057,0
2149,99C,2,-1060000000,0
2151,99P,2,112689973,0


### fy2026_modified


In [45]:
fy2026_modified_cols = ['Agency Number', 'Unit Appropriation Number', "Current Modified Budget Amount", "Current Modified Budget Position"]
fy2026_modified_pubdate = 20260512
fy2026_modified = df[df["Publication Date"] == fy2026_modified_pubdate][fy2026_modified_cols].rename(
    columns={
        "Current Modified Budget Amount":"fy2026_modified_budget_amount",
        "Current Modified Budget Position":"fy2026_modified_position",
        }
    )
fy2026_modified

,Agency Number,Unit Appropriation Number,fy2026_modified_budget_amount,fy2026_modified_position
0,002,20,40316729,319
3,002,21,4459520,0
6,002,40,54732488,452
9,002,41,11537599,0
14,002,61,17866722,193
...,...,...,...,...
2137,944,1,673895,8
2140,944,2,20713,0
2143,945,1,617545,5
2146,945,2,56057,0


### fy2026_adopted


In [46]:
fy2026_adopted_cols = ['Agency Number', 'Unit Appropriation Number', "Adopted Budget Amount", "Adopted Budget Position"]
fy2026_adopted_pubdate = 20260512
fy2026_adopted = df[df["Publication Date"] == fy2026_adopted_pubdate][fy2026_adopted_cols].rename(
    columns={
        "Adopted Budget Amount":"fy2026_adopted_budget_amount",
        "Adopted Budget Position":"fy2026_adopted_position",
        }
    )
fy2026_adopted

,Agency Number,Unit Appropriation Number,fy2026_adopted_budget_amount,fy2026_adopted_position
0,002,20,40515979,319
3,002,21,4237982,0
6,002,40,55440649,455
9,002,41,11306577,0
14,002,61,16885354,173
...,...,...,...,...
2137,944,1,678895,8
2140,944,2,15713,0
2143,945,1,617545,5
2146,945,2,56057,0


## Combine all dataframes


In [47]:
collapsed_df = base_df.merge(right=fy2026_adopted,on=['Agency Number', 'Unit Appropriation Number'], how='left')
collapsed_df = collapsed_df.merge(right=fy2026_modified,on=['Agency Number', 'Unit Appropriation Number'], how='left')
collapsed_df = collapsed_df.merge(right=fy2027_prelim,on=['Agency Number', 'Unit Appropriation Number'], how='left')
collapsed_df = collapsed_df.merge(right=fy2027_exec,on=['Agency Number', 'Unit Appropriation Number'], how='left')
collapsed_df = collapsed_df.fillna(0)
# collapsed_df = collapsed_df.dropna(subset=[
#     "fy2026_adopted_budget_amount",
#     "fy2026_modified_budget_amount",
#     "fy2027_prelim_budget_amount",
#     "fy2027_exec_budget_amount"
# ],how="all")

In [48]:
collapsed_df

,Agency Number,Agency Name,Unit Appropriation Number,Unit Appropriation Name,fy2026_adopted_budget_amount,fy2026_adopted_position,fy2026_modified_budget_amount,fy2026_modified_position,fy2027_prelim_budget_amount,fy2027_prelim_position,fy2027_exec_budget_amount,fy2027_exec_position
0,002,MAYORALTY,20,OFFICE OF THE MAYOR-PS,40515979.0,319.0,40316729.0,319.0,4.046508e+07,318.0,46751202.0,322.0
1,002,MAYORALTY,21,OFFICE OF THE MAYOR-OTPS,4237982.0,0.0,4459520.0,0.0,4.489982e+06,0.0,5031799.0,0.0
2,002,MAYORALTY,40,OFFICE OF MGMT AND BUDGET-PS,55440649.0,455.0,54732488.0,452.0,5.606719e+07,467.0,55900321.0,467.0
3,002,MAYORALTY,41,OFFICE OF MGMT AND BUDGET-OTPS,11306577.0,0.0,11537599.0,0.0,1.134907e+07,0.0,11666672.0,0.0
4,002,MAYORALTY,50,CRIMINAL JUSTICE PROGRAMS PS,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
753,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2,OTHER THAN PERSONAL SERVICES,56057.0,0.0,56057.0,0.0,3.105700e+04,0.0,35105.0,0.0
754,99C,CITYWIDE SAVINGS INITIATIVES,2,Citywide Savings - OTPS,0.0,0.0,0.0,0.0,-1.060000e+09,0.0,0.0,0.0
755,99D,PRIOR YEAR PAYABLES,2,OTHER THAN PERSONAL SERVICES,0.0,0.0,0.0,0.0,0.000000e+00,0.0,-400000000.0,0.0
756,99P,ENERGY ADJUSTMENT,2,OTHER THAN PERSONAL SERVICES,0.0,0.0,0.0,0.0,1.126900e+08,0.0,0.0,0.0


In [49]:
# # Save to CSV
# output_filename = "./Docs/Collapsed_UoA_Budget_Data.csv"
# collapsed_df.to_csv(output_filename, index=False)

### Compute Changes To Exec


In [50]:
change_df = collapsed_df.copy()

In [51]:
change_df["26-27 Budget Change Adopted vs Exec"] = change_df["fy2027_exec_budget_amount"] - change_df["fy2026_adopted_budget_amount"]
change_df["26-27 % Budget Change Adopted vs Exec"] = (change_df["fy2027_exec_budget_amount"] - change_df["fy2026_adopted_budget_amount"])/change_df["fy2026_adopted_budget_amount"]

change_df["26-27 Budget Change Modified vs Exec"] = change_df["fy2027_exec_budget_amount"] - change_df["fy2026_modified_budget_amount"]
change_df["26-27 % Budget Change Modified vs Exec"] = (change_df["fy2027_exec_budget_amount"] - change_df["fy2026_modified_budget_amount"])/change_df["fy2026_modified_budget_amount"]

change_df["27 Budget Change Prelim vs Exec"] = change_df["fy2027_exec_budget_amount"] - change_df["fy2027_prelim_budget_amount"]
change_df["27 % Budget Change Prelim vs Exec"] = (change_df["fy2027_exec_budget_amount"] - change_df["fy2027_prelim_budget_amount"])/change_df["fy2027_prelim_budget_amount"]

# change_df = change_df.fillna(0)#.replace([np.inf, -np.inf], np.nan)

change_df[[
    "26-27 Budget Change Adopted vs Exec",
    "26-27 % Budget Change Adopted vs Exec",
    "26-27 Budget Change Modified vs Exec",
    "26-27 % Budget Change Modified vs Exec",
    "27 Budget Change Prelim vs Exec",
    "27 % Budget Change Prelim vs Exec"
]]

,26-27 Budget Change Adopted vs Exec,26-27 % Budget Change Adopted vs Exec,26-27 Budget Change Modified vs Exec,26-27 % Budget Change Modified vs Exec,27 Budget Change Prelim vs Exec,27 % Budget Change Prelim vs Exec
0,6235223.0,0.153895,6434473.0,0.159598,6.286117e+06,0.155347
1,793817.0,0.187310,572279.0,0.128327,5.418170e+05,0.120672
2,459672.0,0.008291,1167833.0,0.021337,-1.668730e+05,-0.002976
3,360095.0,0.031848,129073.0,0.011187,3.176020e+05,0.027985
4,0.0,NaN,0.0,NaN,0.000000e+00,NaN
...,...,...,...,...,...,...
753,-20952.0,-0.373762,-20952.0,-0.373762,4.048000e+03,0.130341
754,0.0,NaN,0.0,NaN,1.060000e+09,-1.000000
755,-400000000.0,-inf,-400000000.0,-inf,-4.000000e+08,-inf
756,0.0,NaN,0.0,NaN,-1.126900e+08,-1.000000


In [52]:
change_df["26-27 Position Change Adopted vs Exec"] = change_df["fy2027_exec_position"] - change_df["fy2026_adopted_position"]
change_df["26-27 % Position Change Adopted vs Exec"] = (change_df["fy2027_exec_position"] - change_df["fy2026_adopted_position"])/change_df["fy2026_adopted_position"]

change_df["26-27 Position Change Modified vs Exec"] = change_df["fy2027_exec_position"] - change_df["fy2026_modified_position"]
change_df["26-27 % Position Change Modified vs Exec"] = (change_df["fy2027_exec_position"] - change_df["fy2026_modified_position"])/change_df["fy2026_modified_position"]

change_df["27 Position Change Prelim vs Exec"] = change_df["fy2027_exec_position"] - change_df["fy2027_prelim_position"]
change_df["27 % Position Change Prelim vs Exec"] = (change_df["fy2027_exec_position"] - change_df["fy2027_prelim_position"])/change_df["fy2027_prelim_position"]

# change_df = change_df.fillna(0)#.replace([np.inf, -np.inf], np.nan)

change_df[[
    "26-27 Position Change Adopted vs Exec",
    "26-27 % Position Change Adopted vs Exec",
    "26-27 Position Change Modified vs Exec",
    "26-27 % Position Change Modified vs Exec",
    "27 Position Change Prelim vs Exec",
    "27 % Position Change Prelim vs Exec"
]]

,26-27 Position Change Adopted vs Exec,26-27 % Position Change Adopted vs Exec,26-27 Position Change Modified vs Exec,26-27 % Position Change Modified vs Exec,27 Position Change Prelim vs Exec,27 % Position Change Prelim vs Exec
0,3.0,0.009404,3.0,0.009404,4.0,0.012579
1,0.0,NaN,0.0,NaN,0.0,NaN
2,12.0,0.026374,15.0,0.033186,0.0,0.000000
3,0.0,NaN,0.0,NaN,0.0,NaN
4,0.0,NaN,0.0,NaN,0.0,NaN
...,...,...,...,...,...,...
753,0.0,NaN,0.0,NaN,0.0,NaN
754,0.0,NaN,0.0,NaN,0.0,NaN
755,0.0,NaN,0.0,NaN,0.0,NaN
756,0.0,NaN,0.0,NaN,0.0,NaN


In [53]:
# col_order = [
#     'Agency Number',
#     'Agency Name',
#     'Unit Appropriation Number',
#     'Unit Appropriation Name',
#     'fy2026_adopted_budget_amount',
#     'fy2026_adopted_position',
#     '26-27 Budget Change Adopted vs Exec',
#     '26-27 % Budget Change Adopted vs Exec',
#     '26-27 Position Change Adopted vs Exec',
#     '26-27 % Position Change Adopted vs Exec',
#     'fy2026_modified_budget_amount',
#     'fy2026_modified_position',
#     '26-27 Budget Change Modified vs Exec',
#     '26-27 % Budget Change Modified vs Exec',
#     '26-27 Position Change Modified vs Exec',
#     '26-27 % Position Change Modified vs Exec',
#     'fy2027_prelim_budget_amount',
#     'fy2027_prelim_position',
#     '27 Budget Change Prelim vs Exec',
#     '27 % Budget Change Prelim vs Exec',
#     '27 Position Change Prelim vs Exec',
#     '27 % Position Change Prelim vs Exec',
#     'fy2027_exec_budget_amount',
#     'fy2027_exec_position',
#  ]

In [54]:
# # change_df = change_df[col_order]

# output_filename = "./Docs/Collapsed_UoA_Budget_Position_Data_with_Changes.csv"
# change_df.to_csv(output_filename, index=False)

# change_df

## Department (Agency) Totals (sum across UoA)


In [62]:
change_df.columns

Index(['Agency Number', 'Agency Name', 'Unit Appropriation Number',
       'Unit Appropriation Name', 'fy2026_adopted_budget_amount',
       'fy2026_adopted_position', 'fy2026_modified_budget_amount',
       'fy2026_modified_position', 'fy2027_prelim_budget_amount',
       'fy2027_prelim_position', 'fy2027_exec_budget_amount',
       'fy2027_exec_position', '26-27 Budget Change Adopted vs Exec',
       '26-27 % Budget Change Adopted vs Exec',
       '26-27 Budget Change Modified vs Exec',
       '26-27 % Budget Change Modified vs Exec',
       '27 Budget Change Prelim vs Exec', '27 % Budget Change Prelim vs Exec',
       '26-27 Position Change Adopted vs Exec',
       '26-27 % Position Change Adopted vs Exec',
       '26-27 Position Change Modified vs Exec',
       '26-27 % Position Change Modified vs Exec',
       '27 Position Change Prelim vs Exec',
       '27 % Position Change Prelim vs Exec'],
      dtype='str')

In [71]:
target_cols = [
    'fy2026_adopted_budget_amount',
    'fy2026_adopted_position', 
    'fy2026_modified_budget_amount',
    'fy2026_modified_position', 
    'fy2027_prelim_budget_amount',
    'fy2027_prelim_position', 
    'fy2027_exec_budget_amount',
    'fy2027_exec_position'
]

dept_collapsed_df = change_df[[
    'Agency Number',
    *target_cols
    ]].copy()
# dept_collapsed_df = collapsed_df.copy()

# Aggregate
dept_collapsed_df = dept_collapsed_df.groupby(by=["Agency Number"]).sum().reset_index()


# Add Agency Name back to dataframe, reordered
dept_collapsed_df["Agency Name"] = dept_collapsed_df['Agency Number'].apply(lambda num: AGENCY_MAP.get(num))
dept_collapsed_df = dept_collapsed_df[['Agency Number', 'Agency Name',*target_cols]]

dept_collapsed_df

,Agency Number,Agency Name,fy2026_adopted_budget_amount,fy2026_adopted_position,fy2026_modified_budget_amount,fy2026_modified_position,fy2027_prelim_budget_amount,fy2027_prelim_position,fy2027_exec_budget_amount,fy2027_exec_position
0,002,MAYORALTY,196146351.0,1289.0,197354723.0,1306.0,1.881795e+08,1311.0,199358009.0,1332.0
1,003,BOARD OF ELECTIONS,146875148.0,517.0,216647571.0,517.0,1.468751e+08,517.0,206972679.0,517.0
2,004,CAMPAIGN FINANCE BOARD,109481237.0,258.0,123881237.0,258.0,1.347705e+07,87.0,104093091.0,258.0
3,008,OFFICE OF THE ACTUARY,7614099.0,42.0,7677099.0,42.0,7.617044e+06,42.0,8483139.0,42.0
4,010,BOROUGH PRESIDENT - MANHATTAN,6017382.0,56.0,6194656.0,56.0,5.586632e+06,56.0,6079019.0,56.0
...,...,...,...,...,...,...,...,...,...,...
144,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,673602.0,5.0,673602.0,5.0,6.555800e+05,5.0,657628.0,5.0
145,99C,CITYWIDE SAVINGS INITIATIVES,0.0,0.0,0.0,0.0,-1.060000e+09,0.0,0.0,0.0
146,99D,PRIOR YEAR PAYABLES,0.0,0.0,0.0,0.0,0.000000e+00,0.0,-400000000.0,0.0
147,99P,ENERGY ADJUSTMENT,0.0,0.0,0.0,0.0,1.126900e+08,0.0,0.0,0.0


In [72]:
# dept_collapsed_df["26-27 Change Adopted vs Exec"] = dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_adopted_budget_amount"]
# # dept_collapsed_df["26-27 % Change Adopted vs Exec"] = (dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_adopted_budget_amount"])/dept_collapsed_df["fy2026_adopted_budget_amount"]

# dept_collapsed_df["26-27 Change Modified vs Exec"] = dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_modified_budget_amount"]
# # dept_collapsed_df["26-27 % Change Modified vs Exec"] = (dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_modified_budget_amount"])/dept_collapsed_df["fy2026_modified_budget_amount"]

# dept_collapsed_df["27 Change Prelim vs Exec"] = dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2027_prelim_budget_amount"]
# # dept_collapsed_df["27 % Change Prelim vs Exec"] = (dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2027_prelim_budget_amount"])/dept_collapsed_df["fy2027_prelim_budget_amount"]

# dept_collapsed_df = dept_collapsed_df.fillna(0)#.replace([np.inf, -np.inf], np.nan)

# dept_collapsed_df[[
#     "26-27 Change Adopted vs Exec",
#     # "26-27 % Change Adopted vs Exec",
#     "26-27 Change Modified vs Exec",
#     # "26-27 % Change Modified vs Exec",
#     "27 Change Prelim vs Exec",
#     # "27 % Change Prelim vs Exec"
# ]]

In [77]:
dept_collapsed_df["26-27 Budget Change Adopted vs Exec"] = dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_adopted_budget_amount"]
dept_collapsed_df["26-27 % Budget Change Adopted vs Exec"] = (dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_adopted_budget_amount"])/dept_collapsed_df["fy2026_adopted_budget_amount"]

dept_collapsed_df["26-27 Budget Change Modified vs Exec"] = dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_modified_budget_amount"]
dept_collapsed_df["26-27 % Budget Change Modified vs Exec"] = (dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2026_modified_budget_amount"])/dept_collapsed_df["fy2026_modified_budget_amount"]

dept_collapsed_df["27 Budget Change Prelim vs Exec"] = dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2027_prelim_budget_amount"]
dept_collapsed_df["27 % Budget Change Prelim vs Exec"] = (dept_collapsed_df["fy2027_exec_budget_amount"] - dept_collapsed_df["fy2027_prelim_budget_amount"])/dept_collapsed_df["fy2027_prelim_budget_amount"]

# dept_collapsed_df = dept_collapsed_df.fillna(0)#.replace([np.inf, -np.inf], np.nan)

dept_collapsed_df["26-27 Position Change Adopted vs Exec"] = dept_collapsed_df["fy2027_exec_position"] - dept_collapsed_df["fy2026_adopted_position"]
# dept_collapsed_df["26-27 % Position Change Adopted vs Exec"] = (dept_collapsed_df["fy2027_exec_position"] - dept_collapsed_df["fy2026_adopted_position"])/dept_collapsed_df["fy2026_adopted_position"]

dept_collapsed_df["26-27 Position Change Modified vs Exec"] = dept_collapsed_df["fy2027_exec_position"] - dept_collapsed_df["fy2026_modified_position"]
# dept_collapsed_df["26-27 % Position Change Modified vs Exec"] = (dept_collapsed_df["fy2027_exec_position"] - dept_collapsed_df["fy2026_modified_position"])/dept_collapsed_df["fy2026_modified_position"]

dept_collapsed_df["27 Position Change Prelim vs Exec"] = dept_collapsed_df["fy2027_exec_position"] - dept_collapsed_df["fy2027_prelim_position"]
# dept_collapsed_df["27 % Position Change Prelim vs Exec"] = (dept_collapsed_df["fy2027_exec_position"] - dept_collapsed_df["fy2027_prelim_position"])/dept_collapsed_df["fy2027_prelim_position"]

# dept_collapsed_df = dept_collapsed_df.fillna(0)#.replace([np.inf, -np.inf], np.nan)

dept_collapsed_df[[
    "26-27 Budget Change Adopted vs Exec",
    # "26-27 % Budget Change Adopted vs Exec",
    "26-27 Budget Change Modified vs Exec",
    # "26-27 % Budget Change Modified vs Exec",
    "27 Budget Change Prelim vs Exec",
    # "27 % Budget Change Prelim vs Exec",
    "26-27 Position Change Adopted vs Exec",
    # "26-27 % Position Change Adopted vs Exec",
    "26-27 Position Change Modified vs Exec",
    # "26-27 % Position Change Modified vs Exec",
    "27 Position Change Prelim vs Exec",
    # "27 % Position Change Prelim vs Exec"
]]

,26-27 Budget Change Adopted vs Exec,26-27 Budget Change Modified vs Exec,27 Budget Change Prelim vs Exec,26-27 Position Change Adopted vs Exec,26-27 Position Change Modified vs Exec,27 Position Change Prelim vs Exec
0,3211658.0,2003286.0,1.117851e+07,43.0,26.0,21.0
1,60097531.0,-9674892.0,6.009753e+07,0.0,0.0,0.0
2,-5388146.0,-19788146.0,9.061604e+07,0.0,0.0,171.0
3,869040.0,806040.0,8.660950e+05,0.0,0.0,0.0
4,61637.0,-115637.0,4.923870e+05,0.0,0.0,0.0
...,...,...,...,...,...,...
144,-15974.0,-15974.0,2.048000e+03,0.0,0.0,0.0
145,0.0,0.0,1.060000e+09,0.0,0.0,0.0
146,-400000000.0,-400000000.0,-4.000000e+08,0.0,0.0,0.0
147,0.0,0.0,-1.126900e+08,0.0,0.0,0.0


In [79]:
for  c in dept_collapsed_df.columns:
    print(c)

Agency Number
Agency Name
fy2026_adopted_budget_amount
fy2026_adopted_position
fy2026_modified_budget_amount
fy2026_modified_position
fy2027_prelim_budget_amount
fy2027_prelim_position
fy2027_exec_budget_amount
fy2027_exec_position
26-27 Budget Change Adopted vs Exec
26-27 Budget Change Modified vs Exec
27 Budget Change Prelim vs Exec
26-27 Position Change Adopted vs Exec
26-27 Position Change Modified vs Exec
27 Position Change Prelim vs Exec
26-27 % Budget Change Adopted vs Exec
26-27 % Budget Change Modified vs Exec
27 % Budget Change Prelim vs Exec


In [81]:
dept_collapsed_df =  dept_collapsed_df[[
"Agency Number",
"Agency Name",

"27 Budget Change Prelim vs Exec",
"27 % Budget Change Prelim vs Exec",
"26-27 Budget Change Adopted vs Exec",
"26-27 % Budget Change Adopted vs Exec",

"fy2027_exec_budget_amount",
"fy2027_prelim_budget_amount",
"fy2026_adopted_budget_amount",
"fy2026_modified_budget_amount",

"26-27 Budget Change Modified vs Exec",
"26-27 % Budget Change Modified vs Exec",

"27 Position Change Prelim vs Exec",
"26-27 Position Change Adopted vs Exec",
"26-27 Position Change Modified vs Exec",

"fy2026_adopted_position",
"fy2026_modified_position",
"fy2027_prelim_position",
"fy2027_exec_position",

]]

In [84]:
output_filename = "./Docs/Collapsed_Agency_Budget_Position_Data_with_Changes.xlsx"
dept_collapsed_df.to_excel(output_filename, index=False)

dept_collapsed_df

,Agency Number,Agency Name,27 Budget Change Prelim vs Exec,27 % Budget Change Prelim vs Exec,26-27 Budget Change Adopted vs Exec,26-27 % Budget Change Adopted vs Exec,fy2027_exec_budget_amount,fy2027_prelim_budget_amount,fy2026_adopted_budget_amount,fy2026_modified_budget_amount,26-27 Budget Change Modified vs Exec,26-27 % Budget Change Modified vs Exec,27 Position Change Prelim vs Exec,26-27 Position Change Adopted vs Exec,26-27 Position Change Modified vs Exec,fy2026_adopted_position,fy2026_modified_position,fy2027_prelim_position,fy2027_exec_position
0,002,MAYORALTY,1.117851e+07,0.059403,3211658.0,0.016374,199358009.0,1.881795e+08,196146351.0,197354723.0,2003286.0,0.010151,21.0,43.0,26.0,1289.0,1306.0,1311.0,1332.0
1,003,BOARD OF ELECTIONS,6.009753e+07,0.409174,60097531.0,0.409174,206972679.0,1.468751e+08,146875148.0,216647571.0,-9674892.0,-0.044657,0.0,0.0,0.0,517.0,517.0,517.0,517.0
2,004,CAMPAIGN FINANCE BOARD,9.061604e+07,6.723730,-5388146.0,-0.049215,104093091.0,1.347705e+07,109481237.0,123881237.0,-19788146.0,-0.159735,171.0,0.0,0.0,258.0,258.0,87.0,258.0
3,008,OFFICE OF THE ACTUARY,8.660950e+05,0.113705,869040.0,0.114136,8483139.0,7.617044e+06,7614099.0,7677099.0,806040.0,0.104993,0.0,0.0,0.0,42.0,42.0,42.0,42.0
4,010,BOROUGH PRESIDENT - MANHATTAN,4.923870e+05,0.088137,61637.0,0.010243,6079019.0,5.586632e+06,6017382.0,6194656.0,-115637.0,-0.018667,0.0,0.0,0.0,56.0,56.0,56.0,56.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144,945,PUBLIC ADMINISTRATOR-RICHMOND COUNTY,2.048000e+03,0.003124,-15974.0,-0.023714,657628.0,6.555800e+05,673602.0,673602.0,-15974.0,-0.023714,0.0,0.0,0.0,5.0,5.0,5.0,5.0
145,99C,CITYWIDE SAVINGS INITIATIVES,1.060000e+09,-1.000000,0.0,NaN,0.0,-1.060000e+09,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
146,99D,PRIOR YEAR PAYABLES,-4.000000e+08,-inf,-400000000.0,-inf,-400000000.0,0.000000e+00,0.0,0.0,-400000000.0,-inf,0.0,0.0,0.0,0.0,0.0,0.0,0.0
147,99P,ENERGY ADJUSTMENT,-1.126900e+08,-1.000000,0.0,NaN,0.0,1.126900e+08,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
